# Taller: Crear un espacio Genie y probar consultas (Parte 2)

Objetivos:
- Crear un espacio Genie desde la UI (recomendado) y alternativa por código (opcional).
- Conectar Genie a la tabla de inventario creada en la Parte 1.
- Definir instrucciones (indicaciones) para que Genie responda en español y con contexto.
- Probar preguntas y solicitar visualizaciones.

Prerrequisito: Haber creado el catálogo/esquema/tabla `inventario_insumos_oficina` en la Parte 1.



In [ ]:
# Configuración rápida (usa el mismo catálogo/esquema/tabla de la Parte 1)
try:
    dbutils.widgets.text("apellido", "apellido", "Tu apellido")
    APELLIDO = dbutils.widgets.get("apellido").strip().lower()
except Exception:
    APELLIDO = "apellido"

CATALOGO = f"databricks_workshop_{APELLIDO}"
ESQUEMA = "gold"
TABLA = "inventario_insumos_oficina"

spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql(f"USE `{CATALOGO}`.`{ESQUEMA}`")
print(f"Contexto: {CATALOGO}.{ESQUEMA}.{TABLA}")


## A. Crear Genie desde la UI (recomendado)
1. Ve a la vista SQL (DBSQL) en el menú superior.
2. Abre “Genie” (o Assistant) en la barra lateral.
3. Crea un nuevo espacio Genie y asígnale un nombre (ej.: “Genie Inventario Oficina - Apellido).
4. Fuente de datos: selecciona tu catálogo/esquema y elige la tabla `inventario_insumos_oficina`.
5. Instrucciones (System prompt) sugeridas:
   - “Responde en español.”
   - “Cuando cites datos, usa los campos y sus descripciones tal como están en la tabla.”
   - “Si la pregunta es ambigua, solicita aclaraciones y sugiere filtros como fecha o categoría.”
   - “Si corresponde, sugiere visualizaciones (barras/series) y limita resultados.
6. Guarda y prueba el asistente.



## B. (Opcional) Crear/actualizar Genie por código (REST API)
Nota: La API puede variar según la versión. Este ejemplo ilustra el patrón general.

1) Crea un token PAT y ten a mano tu `host` del workspace.
2) Usa la API de Genie/Assistant (cuando esté disponible) o los endpoints de DBSQL para espacios.

Ejemplo (boceto):
```python
import requests, json, os

host = os.environ.get("DATABRICKS_HOST", "https://<tu-workspace>")
token = os.environ.get("DATABRICKS_TOKEN", "<PAT>")

headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

payload = {
  "name": "Genie Inventario Oficina",
  "instructions": "Responde en español, usa la tabla inventario_insumos_oficina.",
  "data_sources": [{
    "catalog": "databricks_workshop_apellido",
    "schema": "gold",
    "tables": ["inventario_insumos_oficina"]
  }]
}

# Ejemplo de endpoint (ilustrativo; valida en tu workspace):
# resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, data=json.dumps(payload))
# print(resp.status_code, resp.text)
```



## C. 5 preguntas de prueba para Genie
1. ¿Cuál es el total de ítems disponibles en el inventario?
2. ¿Cuáles son las 5 subcategorías con mayor stock actual? Muestra un gráfico de barras.
3. ¿Cuál es el promedio de días de rotación por categoría? Muestra una tabla ordenada descendentemente.
4. ¿Qué insumos están por debajo del stock mínimo? Devuélveme los 10 más críticos.
5. ¿Puedes mostrar una serie temporal de las fechas de última compra (conteo por mes) y explicarme tendencias?



## D. Explicación de JOINs con ejemplos

Para enriquecer el inventario, crearemos una tabla pequeña de proveedores y mostraremos distintos JOINs:
- INNER JOIN: Solo filas que hacen match en ambas tablas.
- LEFT JOIN: Todas las filas de la izquierda (inventario) y las que empatan de proveedores (si existen).
- RIGHT/FULL JOIN: Complementan el LEFT para cubrir todos los casos.
- SEMI/ANTI JOIN: Filas que sí tienen match (SEMI) o que no tienen match (ANTI) sin duplicar columnas.



In [ ]:
-- Crear tabla de proveedores de ejemplo
CREATE TABLE IF NOT EXISTS `${CATALOGO}`.`${ESQUEMA}`.`proveedores_info` (
  proveedor STRING,
  pais STRING,
  sla_dias INT,
  rating DOUBLE
) COMMENT 'Tabla de referencia de proveedores para ejemplos de JOIN';

-- Poblar algunos registros (idempotente)
INSERT INTO `${CATALOGO}`.`${ESQUEMA}`.`proveedores_info`
SELECT * FROM VALUES
  ('OfiMax','MX',7,4.5),
  ('Papelería Centro','AR',10,4.0),
  ('TechPlus','US',5,4.7),
  ('Distribuidora Sur','CL',9,4.1)
ON CONFLICT(proveedor) DO NOTHING;


In [ ]:
-- INNER JOIN: solo coincidencias
SELECT i.item_id, i.nombre, i.proveedor, p.pais, p.sla_dias, p.rating
FROM `${CATALOGO}`.`${ESQUEMA}`.`${TABLA}` i
INNER JOIN `${CATALOGO}`.`${ESQUEMA}`.`proveedores_info` p
  ON i.proveedor = p.proveedor
LIMIT 20;


In [ ]:
-- LEFT JOIN: conserva todo el inventario, añade info del proveedor si existe
SELECT i.item_id, i.nombre, i.proveedor, p.pais, p.sla_dias, p.rating
FROM `${CATALOGO}`.`${ESQUEMA}`.`${TABLA}` i
LEFT JOIN `${CATALOGO}`.`${ESQUEMA}`.`proveedores_info` p
  ON i.proveedor = p.proveedor
ORDER BY p.rating DESC NULLS LAST
LIMIT 20;


In [ ]:
-- FULL OUTER JOIN: todas las filas de ambas tablas
SELECT coalesce(i.proveedor, p.proveedor) as proveedor,
       count(i.item_id) as items,
       max(p.rating) as rating
FROM `${CATALOGO}`.`${ESQUEMA}`.`${TABLA}` i
FULL OUTER JOIN `${CATALOGO}`.`${ESQUEMA}`.`proveedores_info` p
  ON i.proveedor = p.proveedor
GROUP BY coalesce(i.proveedor, p.proveedor)
ORDER BY items DESC NULLS LAST
LIMIT 20;


In [ ]:
-- SEMI / ANTI JOIN: existencia o ausencia de match
-- Ítems con proveedor presente en la tabla de referencia
SELECT i.item_id, i.proveedor
FROM `${CATALOGO}`.`${ESQUEMA}`.`${TABLA}` i
SEMI JOIN `${CATALOGO}`.`${ESQUEMA}`.`proveedores_info` p
  ON i.proveedor = p.proveedor
LIMIT 20;

-- Ítems con proveedor NO presente en la tabla de referencia
SELECT i.item_id, i.proveedor
FROM `${CATALOGO}`.`${ESQUEMA}`.`${TABLA}` i
ANTI JOIN `${CATALOGO}`.`${ESQUEMA}`.`proveedores_info` p
  ON i.proveedor = p.proveedor
LIMIT 20;


## E. Expresiones y consultas SQL útiles
Algunas expresiones comunes para que Genie (y tú) aprovechen:
- CASE WHEN para clasificar: p. ej., "stock bajo", "stock medio", "stock alto".
- Funciones de fecha: date_trunc, month, year.
- Ventanas (WINDOW): rank/row_number por categoría/subcategoría.
- Agregaciones: sum, avg, count.



In [ ]:
-- Clasificar nivel de stock y ranking dentro de categoría
WITH base AS (
  SELECT 
    categoria,
    subcategoria,
    item_id,
    nombre,
    stock_actual,
    stock_minimo,
    CASE 
      WHEN stock_actual < stock_minimo THEN 'bajo'
      WHEN stock_actual <= stock_minimo * 1.5 THEN 'medio'
      ELSE 'alto'
    END AS nivel_stock,
    date_trunc('month', fecha_ultima_compra) AS mes_compra
  FROM `${CATALOGO}`.`${ESQUEMA}`.`${TABLA}`
)
SELECT 
  categoria,
  subcategoria,
  item_id,
  nombre,
  nivel_stock,
  stock_actual,
  stock_minimo,
  ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY stock_actual DESC) AS rn_cat,
  COUNT(*) OVER (PARTITION BY categoria, subcategoria) AS items_subcat,
  mes_compra
FROM base
ORDER BY categoria, rn_cat
LIMIT 50;


## F. Benchmarks para probar Genie
Usa las siguientes preguntas de validación. Para cada una, verifica:
- Columnas devueltas y tipos razonables
- Filtros aplicados correctamente
- Ordenación/agrupación coherente
- Si pide gráfico, que lo sugiera/retorne

Preguntas y criterios:
1) "Total de ítems en inventario y promedio de stock por categoría".
   - Esperado: columnas [categoria, total_items, stock_promedio].
2) "Top 10 ítems con stock por debajo del mínimo ordenados ascendente".
   - Esperado: [item_id, nombre, stock_actual, stock_minimo] orden ASC, LIMIT 10.
3) "Stock total por proveedor con rating del proveedor, mostrar gráfico de barras".
   - Esperado: JOIN con proveedores_info; [proveedor, stock_total, rating].
4) "Tendencia mensual de compras (conteo por mes) en el último año".
   - Esperado: date_trunc('month', fecha_ultima_compra), COUNT(*).
5) "Subcategorías con mayor rotación (promedio días_rotacion)".
   - Esperado: [subcategoria, avg_dias], orden DESC.
6) "Proveedores sin coincidencia en inventario (anti join)".
   - Esperado: proveedores presentes solo en proveedores_info.
7) "Ítems por nivel de stock (bajo/medio/alto)".
   - Esperado: usa CASE WHEN similar al ejemplo anterior.
8) "Ítems y ubicación (almacén/pasillo/estante/nivel) filtrados por categoría = 'Escritura'".
   - Esperado: aplica filtro y retorna columnas de ubicación.

Sugerencia: Ejecuta cada pregunta y valida la respuesta; si difiere, pide a Genie que corrija filtros/agrupación/orden.



## G. Añadir descripción del espacio Genie usando la UI
1. En la vista Genie, abre tu espacio creado.
2. Edita la “Descripción”/“Instructions” para incluir:
   - Contexto: "Este asistente responde sobre el inventario de insumos de oficina."
   - Idioma: "Responde en español."
   - Estilo: "Da respuestas claras y concisas con tablas/resúmenes y sugiere gráficos cuando corresponda."
   - Alcance: "No inventes datos fuera de las tablas configuradas."
3. Guarda los cambios y realiza una pregunta de prueba para validar que la descripción influye en la respuesta.



## H. Calificar respuestas y ver monitoreo
1. En el panel de conversación de Genie, usa los controles de calificación (👍/👎) y añade comentarios sobre exactitud/claridad.
2. Repite con varias preguntas (usa los Benchmarks) para generar señales de calidad.
3. Ve a la sección de monitoreo/telemetría de DBSQL o del Assistant para revisar:
   - Tasa de éxito, tiempos de respuesta, errores.
   - Preguntas más frecuentes y calidad promedio.
4. Opcional: define etiquetas (tags) o categorías en tus pruebas para comparar iteraciones.

